# 2.5.2 Padding マスク と Subsequent マスク

## パディングマスク
**`<pad>` Tokenに対応する位置のAttention重みを0にするためのマスク。**

学習時に複数のTokenベクトル(≒文章)をバッチとして入力する際、Tokenベクトルの次元を合わせるために`<pad>`Tokenが使用される。    
※入力される文章が必ず同じ長さとは限らないので、`<pad>`Tokenを追加して足りない部分を埋める。  

Transformerでの計算においては`<pad>` Tokenを無視したいため、`<pad>` Tokenの位置をTrueとするマスクを作成し、Attention重みを計算したあとにこのマスクを使用することで`<pad>` Tokenの位置のtokenに対応するAttention重みを-∞に置き換える。  
これに対してSoftmax関数を適用すると、`<pad>` Tokenの位置のAttention重みは0になるため、Transformerの計算において`<pad>` Tokenの影響を無視できるようになる。

Transformerの中の全てのAttention計算で使用

In [1]:
import torch
from torch import Tensor

def create_padding_mask(pad_id: int, batch_tokens: Tensor):
    mask = batch_tokens == pad_id  # batch_tokens内の各トークンがpad_idであるかを判定し、True/Falseの行列を作成
    mask = mask.unsqueeze(1) # 次元を追加してマスクの形状を(batch_size, sequence_length)->(batch_size, 1, sequence_length)に変換
    return mask

In [2]:
batch_size = 3
sequence_length = 5

padding_id = 0  #  <pad>トークンを0とする

# 入力トークンの例
tokens = torch.Tensor([[5, 3, 3, padding_id, padding_id], [1, 9, 4, 3, 1], [5, 3, 5, 1, padding_id]]) 
print("tokens: \n", tokens)

# Attention重みの計算
#  入力トークンからAttention重みを計算した結果の想定
score = torch.randn(batch_size, sequence_length, sequence_length) 
print("score shape: \n", score.shape)
print("score: \n", score)  # Attention重みの例を表示([3, 5, 5])

# パディングマスクを作成
#  <pad> Tokenの位置をTrueとするマスクを作成
mask = create_padding_mask(padding_id, tokens)  # i番目のtokenの値が0ならmaskのi番目をTrueを設定
print("mask.shape: \n", mask.shape)  # torch.Size([3, 1, 5])
print("mask: \n", mask)  # 1行目のmaskを表示

# scoreにmaskを適用してmaskがTrueの位置を-infに置き換える
masked_score = score.masked_fill(mask, float("-inf")) 
print("masked_score: \n", masked_score)

# masked_scoreにsoftmaxを適用してAttention重みを計算
#  maskがTrueの位置は-infになっているため、softmaxを適用するとmaskがTrueの位置のAttention重みは0になる
attention_weights = torch.softmax(masked_score, dim=-1) 
print("attention_weights: \n", attention_weights)

tokens: 
 tensor([[5., 3., 3., 0., 0.],
        [1., 9., 4., 3., 1.],
        [5., 3., 5., 1., 0.]])
score shape: 
 torch.Size([3, 5, 5])
score: 
 tensor([[[-1.5657, -0.4910, -0.0919, -0.6900, -0.3515],
         [-0.0043,  0.2203,  0.3336, -0.6735, -0.3025],
         [ 1.2431,  0.5010, -0.4694, -1.2729, -0.0522],
         [-0.5350,  0.5283, -1.1122,  0.7172,  0.7479],
         [-0.8845, -1.4311, -0.9405,  1.0367, -0.2620]],

        [[ 1.0824, -0.6802, -1.7010, -0.6292,  0.0475],
         [-0.2451, -0.7637, -1.5561,  0.2891,  0.9411],
         [-0.6961,  0.0331,  0.0637, -1.5543,  1.5014],
         [ 0.8172,  0.5525, -1.3971,  0.3166,  0.7470],
         [ 2.4986,  0.3985,  0.4285,  0.3292,  0.1384]],

        [[-0.1289, -1.9545, -0.0522, -1.2925, -0.2212],
         [ 0.3946, -2.3740,  0.0535, -0.3682, -0.9211],
         [-0.5276,  0.1069,  0.1443, -0.2929,  0.7772],
         [ 0.8709,  0.9703, -0.0563,  1.0739,  0.0931],
         [-2.1166,  1.2127, -0.1449, -0.4640,  0.4809]]])
mask.sh

## 後継マスク
**特定のTokenの位置よりも後ろのtokenに対応する位置のAttention重みを0にするためのマスク。(デコーダのみ)**

学習時にはデコーダーに対して文章全体が入力される。  
しかし特定のTokenの予測には、そのTokenよりも前のTokenしか参照させてはいけない。(=そのTokenから先のTokenは参照させてはいけない。)
予測対象のTokenから先の文章を参照させないようにする必要がある。  

Transformerでの計算においては特定のTokenよりも後ろのTokenを無視したいため、その位置よりも後ろの要素をTrueとする上三角行列を作成し、Attention重みを計算したあとにこのマスクを使用することで特定のTokenの位置よりも後ろのtokenに対応するAttention重みを-∞に置き換える。  
これに対してSoftmax関数を適用すると、特定のTokenの位置よりも後ろのtokenに対応する位置のAttention重みは0になるため、Transformerの計算において後ろのTokenの影響を無視できるようになる。

Transformerでデコーダーのみに使用される。

In [3]:
def create_subsequent_mask(batch_tokens: Tensor):
    sequence_len = batch_tokens.size(1)
    mask = torch.triu(
        torch.full((sequence_len, sequence_len), 1), # 上三角行列を作成(上三角の要素を1、その他を0とする)
        diagonal=1,
    )

    print("subsequent mask (before unsqueeze): \n", mask)  # 上三角行列を表示

    mask = mask == 1  # 各要素の値が1であるかを判定し、True/Falseの行列を作成
    mask = mask.unsqueeze(0) # 次元を追加してマスクの形状を(sequence_length, sequence_length)->(1, sequence_length, sequence_length)に変換
    return mask

In [4]:
batch_size = 3
sequence_length = 5

# 入力トークンの例
tokens = torch.randn(batch_size, sequence_length)

# Attention重みの計算
#  入力トークンからAttention重みを計算した結果の想定
score = torch.randn(batch_size, sequence_length, sequence_length)
print("score: \n", score)

# 後続マスクを作成 
mask = create_subsequent_mask(tokens)
print("mask.shape: ", mask.shape)  # torch.Size([1, 5, 5])
print("mask: \n", mask) 

# scoreにmaskを適用してmaskがTrueの位置を-infに置き換える
masked_score = score.masked_fill(mask, float("-inf"))
print("masked_score: \n", masked_score)

# masked_scoreにsoftmaxを適用してAttention重みを計算
#  maskがTrueの位置は-infになっているため、softmaxを適用するとmaskがTrueの位置のAttention重みは0になる
attention_weights = torch.softmax(masked_score, dim=-1) 
print("attention_weights: \n", attention_weights)

score: 
 tensor([[[-5.2361e-01,  3.8662e-01,  5.3165e-01, -1.0169e+00, -3.9153e-01],
         [-1.4539e-01, -5.4422e-01, -1.6377e+00,  1.1937e+00,  1.3465e+00],
         [ 8.5080e-01, -2.8256e-01,  1.3970e+00,  3.6603e-01,  4.4322e-01],
         [ 1.0535e+00, -6.9719e-01,  7.8248e-01, -6.7794e-01, -1.6786e-01],
         [-1.4363e+00,  9.3049e-01,  3.4607e-01,  1.5155e+00,  3.1946e-01]],

        [[ 1.6345e-01, -9.3296e-01, -2.8861e-01,  2.0243e-01,  1.0607e+00],
         [-6.1834e-01, -6.1767e-01, -5.2141e-01,  2.9264e-01, -2.4157e-01],
         [ 1.8599e+00,  1.3432e+00, -6.1689e-01, -1.1714e-02, -8.8749e-01],
         [ 4.9805e-01, -1.8285e-01,  7.8312e-01,  1.0668e+00, -6.6080e-01],
         [ 2.7952e+00, -1.4576e-01,  7.8769e-01,  1.8617e+00, -1.9455e-01]],

        [[ 1.5256e+00, -8.3310e-02,  1.5972e-01, -9.2236e-01,  4.6001e-01],
         [ 1.2053e-01,  3.1820e-01, -6.1237e-01, -1.6825e+00, -5.2457e-01],
         [ 2.3334e+00,  4.7510e-01,  4.7158e-01, -6.7281e-01,  6.6300e-04],